# Fitting QENS data with EasyDynamics

Previously, some quasi-elastic neutron scattering (QENS) data has been [simulated](./../3-mcstas/mcstas-qens.ipynb) and [reduced](./../4-reduction/reduction-qens.ipynb), and can now be analysed with [`easydynamics`](https://easyscience.github.io/dynamics-lib/).

In the reduction step we produced two data sets:

- an **elastic** sample, which scatters neutrons without changing their energy. We will use it to measure the **instrument resolution**.
- a **quasi-elastic** sample, in which the scatterers diffuse. This is the data we actually want to interpret.

The workflow is as follows: We first load the data into EasyDynamics and inspect it. We then determine the resolution function using the elastic data. Next, we fit the QENS data to an empirical model and inspect the resulting fit. Finally, we extract physically meaningful information from these fits.

TO DO:
- Look into background, either add background to McStas data or remove background models
- Add quiz questions
- Figure out what's up with discrepancy between simulation and fit

In [ ]:
import numpy as np

import easydynamics as edyn
from easydynamics.experiment import Experiment
import easydynamics.sample_model as sm

# Quiz questions for this notebook
from jupyterquiz import display_quiz
from quizlib import qens as quiz

# Make the plots interactive
%matplotlib widget

## Load and prepare the data

First we load the **elastic** sample data. We will use it to determine the instrument resolution, in the same way that a vanadium measurement is used in a real experiment.

Our reduced data are histograms of neutron counts, and they need a small adjustment before fitting. Future versions of EasyDynamics will not need this step.

Some of the bins have zero counts, and consequently zero variance. The weight of each point when fitting the data is 1/sqrt(variance), which therefore diverges, giving those bins infinite weight and destabilising the fit. The solution that we will employ is to give all these bins a variance of 1 instead. 


In [ ]:
def prepare_data(experiment):
    """Make the counts data usable by the fit: use uniform weights and drop the unit."""
    data = experiment.data
    indices=data.variances<=0.5
    data.variances[indices] = 1.0  
    data.unit="dimensionless"
    experiment.data = data


In [ ]:
filename = '../4-reduction/energy_QENS_elastic.h5'

elastic_experiment = Experiment('Elastic')
elastic_experiment.load_hdf5(filename=filename)
prepare_data(elastic_experiment)

elastic_experiment.plot_data(slicer=True)

Next we load the **quasi-elastic** sample data. This is the data whose dynamics we want to understand.

In [ ]:
filename2 = '../4-reduction/energy_QENS_sample.h5'

qe_experiment = Experiment('QuasiElastic')
qe_experiment.load_hdf5(filename=filename2)
prepare_data(qe_experiment)


qe_experiment.plot_data(slicer=True)

⚠️ **If you did not complete the QENS data reduction yesterday**, you can use some pre-prepared data by uncommenting and running the cell below.

In [ ]:
# import utils
# filename = utils.fetch_data('4-reduction/energy_QENS_elastic.h5')
# elastic_experiment = Experiment('Elastic')
# elastic_experiment.load_hdf5(filename=filename)

# filename2 = utils.fetch_data('4-reduction/energy_QENS_sample.h5')
# qe_experiment = Experiment('QuasiElastic')
# qe_experiment.load_hdf5(filename=filename2)

## Step 1: Determine the resolution from the elastic sample

The scattering from the elastic sample is, to a very good approximation, a delta function in energy transfer: the neutrons come out with the same energy they went in with. The width we actually measure is therefore entirely due to the **instrument resolution**.

We model the resolution as a single `Gaussian`. We collect it in a `ComponentCollection` and use that to build a `SampleModel`. (In a real experiment the resolution may need several Gaussians or other shapes to be described accurately.)

In [ ]:
resolution_components = sm.ComponentCollection()
res_gauss = sm.Gaussian(width=0.002, area=1, name='Res. Gauss')
resolution_components.append_component(res_gauss)

resolution_sample_model = sm.SampleModel(components=resolution_components)

The background is not exactly zero, so we also add a `BackgroundModel`. We use a `Polynomial` with a single coefficient, i.e. a flat background.

In [ ]:
poly=sm.Polynomial(coefficients=[0.001], name='Background')
poly.coefficients[0].min=0.0
background_model = sm.BackgroundModel(components=poly)

The background model goes into an `InstrumentModel`. This model also contains a fittable energy offset that accounts for any misalignment of the instrument; all components are centred on this offset.

In [ ]:
instrument_model = sm.InstrumentModel(
    background_model=background_model,
)

We now collect everything in an `Analysis` object: a display name, the experiment, the sample model and the instrument model. EasyDynamics automatically generates a model for each `Q` value in the data.

In [ ]:
elastic_analysis = edyn.Analysis(
    display_name='Elastic / Resolution',
    experiment=elastic_experiment,
    sample_model=resolution_sample_model,
    instrument_model=instrument_model,
)

Let us first fit a single `Q` index and plot the data and model to see how it looks. We use the `independent` fit method for one arbitrary `Q` index.

In [ ]:

elastic_analysis.fit(fit_method='independent', Q_index=2)
elastic_analysis.plot_data_and_model(Q_index=2)

The fit looks good, so let us fit all `Q` indices independently and plot the results.

In [ ]:
elastic_analysis.fit(fit_method='independent')
elastic_analysis.plot_data_and_model()

It is useful to inspect the fitted parameters. We can turn them into a scipp dataset, and we can plot any of them as a function of `Q` with `plot_parameters`. A good resolution function should have a width and area that vary only slowly with `Q`.

In [ ]:
elastic_analysis.parameters_to_dataset()

In [ ]:
elastic_analysis.plot_parameters(names=['Res. Gauss width'])

In [ ]:
elastic_analysis.plot_parameters(names=['Res. Gauss area'])

## Step 2: Fit the quasi-elastic sample at each Q

We are now happy with the resolution and can turn to the quasi-elastic sample. Looking at the data we plotted earlier, it has a sharp elastic peak, a broader quasi-elastic peak, and a small flat background.

We describe it with a `SampleModel` containing:

- a `DeltaFunction` for the elastic (immobile) scattering, and
- a `Lorentzian` for the quasi-elastic broadening caused by motion.

In [ ]:
delta_function = sm.DeltaFunction(name='DeltaFunction', area=0.5)
lorentzian = sm.Lorentzian(name='Lorentzian', area=3.5, width=0.015)

component_collection = sm.ComponentCollection(
    components=[delta_function, lorentzian],
)
sample_model = sm.SampleModel(components=component_collection)

poly=sm.Polynomial(coefficients=[0.001])
poly.coefficients[0].min=0.0
background_model = sm.BackgroundModel(components=poly)


We build a new `InstrumentModel`, and this time we give it a resolution: the `sample_model` from our elastic fit. All of its parameters are automatically fixed and the resolution is normalised to have area 1.

In [ ]:
instrument_model = sm.InstrumentModel(
    # background_model=background_model,
    resolution_model=elastic_analysis.sample_model,
)

qe_analysis = edyn.Analysis(
    display_name='Quasi-elastic per-Q',
    experiment=qe_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

`Analysis` handles the convolution of the `sample_model` with the resolution. The calculation is analytical where possible and numerical otherwise. 

Before fitting, it is a good idea to check the start guesses by plotting the data together with the model.

In [ ]:
qe_analysis.plot_data_and_model()

The start guesses look reasonable, so let us fit every `Q` independently and plot the result.

In [ ]:
qe_analysis.fit(fit_method='independent')
qe_analysis.plot_data_and_model()

The interesting parameters are the width and area of the `Lorentzian`. Let us plot them as a function of `Q`.

In [ ]:
qe_analysis.plot_parameters(names=['Lorentzian width'], vmin=0, vmax=0.02, xmin=0, xmax=2.1)

Sometimes, the fitter produces no error bars on the fit parameters. We are working on fixing this, but have to work around it in the meantime.

In [ ]:
def fix_missing_variances(analysis):
    """Fix missing variances on fitted parameters in an analysis."""
    for param in analysis.get_all_parameters():
        if param.variance ==0.0:
            param.variance = 0.00001*param.value

fix_missing_variances(qe_analysis)
qe_analysis.plot_parameters(names=['Lorentzian width'], vmin=0, vmax=0.02, xmin=0, xmax=2.1)

## Step 3: Fit a jump-diffusion model to the widths

The width does not simply grow like $Q^2$: it levels off at high $Q$. This is the signature of **jump diffusion**, in which a particle sits still for a residence time $\tau$ and then jumps to a new site. The half-width of the quasi-elastic Lorentzian is

$$
\Gamma(Q) = \frac{\hbar\,D\,Q^2}{1 + D\,\tau\,Q^2},
$$

where $D$ is the diffusion coefficient and $\tau$ is the residence (relaxation) time. At low $Q$ this reduces to ordinary diffusion, $\Gamma \approx \hbar D Q^2$, while at high $Q$ it saturates at the plateau $\hbar/\tau$.

Our width curve shows exactly this behaviour — a $Q^2$ rise that bends over towards a plateau within the measured range — so the data constrain **both** parameters: the low-$Q$ slope fixes $D$, and the high-$Q$ plateau fixes $\tau$.

As a first step we fit the jump-diffusion model to the fitted Lorentzian parameters we obtained above. We create a `JumpTranslationalDiffusion` model whose `lorentzian_name` matches our fitted `Lorentzian`, wrap it in a `FitBinding`, and pass it to a `ParameterAnalysis` together with the per-`Q` analysis. Because the model's `lorentzian_name` matches the component, the binding fits both the `Lorentzian width` (giving $D$ and $\tau$) and the `Lorentzian area` (giving the scale).

In [ ]:
jump_diffusion_model = sm.JumpTranslationalDiffusion(
    name='Jump Translational Diffusion',
    lorentzian_name='Lorentzian',
    diffusion_coefficient=4.6e-10,
    relaxation_time=22.0,  # ps
    scale=0.5,
)

binding = edyn.FitBinding(model=jump_diffusion_model)

parameter_analysis = edyn.ParameterAnalysis(
    parameters=qe_analysis,
    bindings=binding,
)

We first plot the start guess to see if it is reasonable.

In [ ]:
parameter_analysis.plot(names=['Lorentzian width'], xmin=0, xmax=2.1, vmin=0, vmax=0.02)

Then we fit and plot again.

In [ ]:
parameter_analysis.fit()
parameter_analysis.plot(names=['Lorentzian width'], xmin=0, xmax=2.1, vmin=0, vmax=0.02)

We can read off the fitted jump-diffusion parameters, with uncertainties.

In [ ]:
parameter_analysis.get_all_parameters()

## Step 4: Fit the jump-diffusion model to all the data at once

The two-step approach above works, but we can do better. Now that we know the quasi-elastic scattering follows a jump-diffusion model, we can fit that model **directly to the data**, using all `Q` values simultaneously. This uses every data point at once and generally gives smaller uncertainties. In addition to the diffusion, we still describe the elastic incoherent scattering with a `DeltaFunction`.

We build a new `SampleModel` that has a `DeltaFunction` component and a `JumpTranslationalDiffusion` diffusion model, and new `BackgroundModel` and `InstrumentModel` objects.

In [ ]:
delta_function = sm.DeltaFunction(name='DeltaFunction', area=0.5)
component_collection = sm.ComponentCollection(components=[delta_function])

diffusion_model = sm.JumpTranslationalDiffusion(
    name='Jump Translational Diffusion',
    diffusion_coefficient=2.5e-10,
    relaxation_time=18.0,  # ps
    scale=2.5,
)

sample_model = sm.SampleModel(
    components=component_collection,
    diffusion_models=diffusion_model,
)

poly=sm.Polynomial(coefficients=[0.001])
poly.coefficients[0].min=0.0
background_model = sm.BackgroundModel(components=poly)


In [ ]:
instrument_model = sm.InstrumentModel(
    # background_model=background_model,
    resolution_model=elastic_analysis.sample_model,
    energy_offset = 1e-3
)

diffusion_analysis = edyn.Analysis(
    display_name='Jump Diffusion Full Analysis',
    experiment=qe_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

diffusion_analysis.instrument_model.resolution_model.fix_all_parameters()

As always, we check the start guess before fitting.

In [ ]:
diffusion_analysis.plot_data_and_model()

Now we fit all the data simultaneously.

In [ ]:
diffusion_analysis.fit(fit_method='simultaneous')


In [ ]:
diffusion_analysis.plot_data_and_model(plot_residuals=True,autoscale=False)

The diffusion parameters are just a couple of numbers with uncertainties, so instead of plotting them we display them directly. Both the diffusion coefficient $D$ and the residence time $\tau$ are now determined by the data.

In [ ]:
diffusion_model.get_global_variables()

For reference, here are the parameters from the two-step fit (fitting the Lorentzian widths and areas). Notice that fitting all the data simultaneously generally gives smaller uncertainties than the two-step route.

In [ ]:
parameter_analysis.get_all_variables()

Finally, since we know what we put into McStas we can compare our answers to the true values: The diffusion coefficient was 4.6e-10 and relaxation time 22 ps